# 06 Compute Budget Tradeoff

This notebook compares TOAH and AO under different compute budgets.

Input: `result/table/06_compute_budget_tradeoff.csv`

Outputs: PDF figures in `result/figure/` and aggregated tables in `result/table/`.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path(__file__).resolve().parents[1]
tbl_dir = ROOT / "result" / "table"
fig_dir = ROOT / "result" / "figure"
fig_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(tbl_dir / "06_compute_budget_tradeoff.csv")
df.head()


In [ ]:
# Aggregate mean/std across runs per budget label
metrics = ["avg_sum_queue", "avg_sense_u", "deadline_violation_rate", "avg_slot_ms"]

agg = df.groupby(["budget"])[metrics].agg(["mean", "std"]).reset_index()
agg.columns = ["budget"] + [f"{m}_{s}" for m in metrics for s in ["mean", "std"]]

out_table = tbl_dir / "06_compute_budget_table.csv"
agg.to_csv(out_table, index=False)
print("Saved table:", out_table)

agg


In [ ]:
# Scatter: runtime vs performance (queue)
plt.figure()
plt.scatter(agg["avg_slot_ms_mean"], agg["avg_sum_queue_mean"])
for _, r in agg.iterrows():
    plt.annotate(r["budget"], (r["avg_slot_ms_mean"], r["avg_sum_queue_mean"]))
plt.xlabel("Avg per-slot runtime (ms)")
plt.ylabel("Average total queue")
plt.grid(True, alpha=0.3)
out = fig_dir / "06_scatter_runtime_vs_queue.pdf"
plt.savefig(out, format="pdf", bbox_inches="tight")
print("Saved:", out)


In [ ]:
# Scatter: runtime vs sensing
plt.figure()
plt.scatter(agg["avg_slot_ms_mean"], agg["avg_sense_u_mean"])
for _, r in agg.iterrows():
    plt.annotate(r["budget"], (r["avg_slot_ms_mean"], r["avg_sense_u_mean"]))
plt.xlabel("Avg per-slot runtime (ms)")
plt.ylabel("Average sensing uncertainty $u_t$")
plt.grid(True, alpha=0.3)
out = fig_dir / "06_scatter_runtime_vs_sense_u.pdf"
plt.savefig(out, format="pdf", bbox_inches="tight")
print("Saved:", out)


In [ ]:
# Line: AO performance vs budget, with TOAH as a reference line
toah_row = agg[agg["budget"] == "TOAH-default"].iloc[0]
ao_rows = agg[agg["budget"].str.startswith("AO-")].copy()
ao_rows["steps"] = ao_rows["budget"].str.replace("AO-steps", "", regex=False).astype(int)
ao_rows = ao_rows.sort_values("steps")

plt.figure()
plt.plot(ao_rows["steps"], ao_rows["avg_sum_queue_mean"], marker="o", label="AO")
plt.axhline(toah_row["avg_sum_queue_mean"], linestyle="--", label="TOAH (default)")
plt.xlabel("AO inner steps per slot")
plt.ylabel("Average total queue")
plt.legend()
plt.grid(True, alpha=0.3)
out = fig_dir / "06_line_ao_steps_vs_queue.pdf"
plt.savefig(out, format="pdf", bbox_inches="tight")
print("Saved:", out)

plt.figure()
plt.plot(ao_rows["steps"], ao_rows["avg_sense_u_mean"], marker="o", label="AO")
plt.axhline(toah_row["avg_sense_u_mean"], linestyle="--", label="TOAH (default)")
plt.xlabel("AO inner steps per slot")
plt.ylabel("Average sensing uncertainty $u_t$")
plt.legend()
plt.grid(True, alpha=0.3)
out = fig_dir / "06_line_ao_steps_vs_sense_u.pdf"
plt.savefig(out, format="pdf", bbox_inches="tight")
print("Saved:", out)


In [ ]:
# Bar: deadline violation rate vs budget
labels = list(agg["budget"])
vals = agg["deadline_violation_rate_mean"].values
errs = agg["deadline_violation_rate_std"].values

plt.figure()
plt.bar(range(len(labels)), vals, yerr=errs, capsize=3)
plt.xticks(range(len(labels)), labels, rotation=30, ha="right")
plt.ylabel("Deadline violation rate")
plt.grid(True, axis="y", alpha=0.3)
out = fig_dir / "06_bar_deadline_violation.pdf"
plt.savefig(out, format="pdf", bbox_inches="tight")
print("Saved:", out)
